# CPU preparation-only recovery
Use accelerator None and High-RAM. This skips pilot_inputs verification and uses the existing successful report. Preparation still hashes its source inputs and validates provenance. Existing owned outputs resume; unknown or corrupt outputs fail without overwrite. Run this cell only. Do not run Section A again.


In [ ]:
import os, sys, subprocess, json, tempfile
from pathlib import Path
import torch
assert not torch.cuda.is_available(), 'Switch to a CPU-only runtime (accelerator None) for this preparation phase.'
from google.colab import drive
drive.mount('/content/drive')
REPO = Path(tempfile.mkdtemp(prefix='eeg-patch-pr2-cpu-dictfix-',dir='/content'))/'repo'
subprocess.run(['git','clone','--branch','audit/phase1-reproducible-alignment',
                'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface-hub==0.26.2','psutil==6.1.0','filelock==3.16.1'],check=True)
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
cpu_env = dict(os.environ,CUDA_VISIBLE_DEVICES='')
pointer = json.loads((RAW/'pilot_inputs_latest.json').read_text())
report = json.loads(Path(pointer['report']).read_text())
print('Verification report:',pointer['report'])
print('EEG verification:',report['status'])
print('All image/pooled/EEG inputs ready:',report['ready_for_preparation'])
print('Missing:',report['inventory']['missing'])
assert report['ready_for_preparation'], 'EEG is retained; provide the missing image/cache inputs before continuing.'
WORK = Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP = WORK/'pilot_sub01_200concepts_v1'
prepare_args = [sys.executable,'-m','phase1.patch_prepare','--root',str(RAW),'--cache',str(CACHE),
                '--image-root',report['inventory']['image_root'],'--output',str(PREP),
                '--subjects','sub-01','--concepts','200']
if PREP.exists():
    assert (PREP/'preparation_state.json').is_file(), 'Existing directory is preserved. Choose a new PREP path for verified CPU preparation.'
    prepare_args += ['--resume']
print('Preparation destination:', PREP.resolve(), 'exists:', PREP.exists(),flush=True)
subprocess.run(prepare_args,cwd=REPO,env=cpu_env,check=True)
subprocess.run([sys.executable,'-c',
                'import sys; from phase1.patch_prepare import verify_prepared; verify_prepared(sys.argv[1])',
                str(PREP)],cwd=REPO,env=cpu_env,check=True)
print('CPU preparation finished. STOP: review the report and obtain approval before allocating a GPU.')
